# Notebook 04 — Classification Stretch Goal

**Question:** "Will this engine fail within the next 30 cycles?"

This reformulates the RUL regression problem as binary classification.
We train a LightGBM classifier, evaluate with precision-recall AUC
(preferred over ROC-AUC for imbalanced data), and compare the two
problem framings.

**Primary metric:** AUC-PR (area under precision-recall curve)  
**Why:** The positive class ("fail soon") is a minority (~15–20% of rows),
so AUC-ROC can be misleadingly high.


In [ ]:
import sys; sys.path.insert(0, '..')

import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    precision_recall_curve, average_precision_score,
    roc_auc_score, f1_score, classification_report,
    ConfusionMatrixDisplay,
)
from lightgbm import LGBMClassifier
import mlflow

from src.data import load_raw, attach_rul_labels
from src.features import build_features, get_feature_columns
from src.train import save_model

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

FIGURES_DIR   = Path('../reports/figures')
MODELS_DIR    = Path('../models')
PROCESSED_DIR = Path('../data/processed')

FAIL_HORIZON = 30   # predict failure within this many cycles

print('Imports OK')

---
## 1. Build the classification dataset

In [ ]:
# Load engineered features (same as regression notebook)
train_raw = attach_rul_labels(load_raw('FD001', 'train'), clip=None)  # unclipped
train_feat = build_features(train_raw, dataset='FD001', processed_dir=PROCESSED_DIR)

# Binary label: 1 if RUL <= FAIL_HORIZON
train_feat['fail_30'] = (train_feat['rul'] <= FAIL_HORIZON).astype(int)

n_pos  = train_feat['fail_30'].sum()
n_neg  = len(train_feat) - n_pos
pos_pct = 100 * n_pos / len(train_feat)

print(f'Total rows  : {len(train_feat):,}')
print(f'Positive (fail within {FAIL_HORIZON} cycles): {n_pos:,}  ({pos_pct:.1f}%)')
print(f'Negative : {n_neg:,}  ({100-pos_pct:.1f}%)')
print()
print(f'Class imbalance ratio: 1:{n_neg/n_pos:.1f}  — '
      f'AUC-PR is the right primary metric, not accuracy.')

In [ ]:
feature_cols = get_feature_columns(train_feat)
# Remove fail_30 from features if accidentally included
feature_cols = [c for c in feature_cols if c != 'fail_30']

X = train_feat[feature_cols].values
y = train_feat['fail_30'].values
groups = train_feat['unit'].values

print(f'Feature matrix: {X.shape}')

---
## 2. GroupKFold cross-validation

In [ ]:
gkf = GroupKFold(n_splits=5)

fold_auc_roc = []
fold_auc_pr  = []
fold_f1      = []
oof_prob     = np.zeros(len(y), dtype=float)

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X, y, groups=groups)):
    X_tr, X_val = X[tr_idx], X[val_idx]
    y_tr, y_val = y[tr_idx], y[val_idx]

    clf = LGBMClassifier(
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=63,
        class_weight='balanced',   # handles imbalance
        random_state=42,
        n_jobs=-1,
        verbose=-1,
    )
    clf.fit(X_tr, y_tr)
    probs = clf.predict_proba(X_val)[:, 1]
    oof_prob[val_idx] = probs

    fold_auc_roc.append(roc_auc_score(y_val, probs))
    fold_auc_pr.append(average_precision_score(y_val, probs))
    preds = (probs >= 0.5).astype(int)
    fold_f1.append(f1_score(y_val, preds))

print(f'GroupKFold CV results (5 folds):')
print(f'  AUC-ROC : {np.mean(fold_auc_roc):.3f} ± {np.std(fold_auc_roc):.3f}')
print(f'  AUC-PR  : {np.mean(fold_auc_pr):.3f} ± {np.std(fold_auc_pr):.3f}  ← primary metric')
print(f'  F1      : {np.mean(fold_f1):.3f} ± {np.std(fold_f1):.3f}')

---
## 3. Precision-recall curve and confusion matrix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Precision-recall curve (OOF probabilities)
precision, recall, thresholds = precision_recall_curve(y, oof_prob)
aucpr = average_precision_score(y, oof_prob)

axes[0].plot(recall, precision, 'steelblue', lw=2)
axes[0].axhline(y.mean(), color='tomato', ls='--', lw=1.5,
                label=f'Random baseline ({y.mean():.2f})')
axes[0].fill_between(recall, precision, alpha=0.15, color='steelblue')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title(f'Precision-Recall Curve\nAUC-PR = {aucpr:.3f}')
axes[0].legend()
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1.05])

# Confusion matrix at threshold=0.5
oof_pred = (oof_prob >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(
    y, oof_pred, ax=axes[1],
    display_labels=['No failure', f'Fail ≤ {FAIL_HORIZON} cycles'],
    colorbar=False,
)
axes[1].set_title(f'Confusion Matrix (threshold=0.5, OOF)')

fig.suptitle(
    f'LightGBM classifier — "fail within {FAIL_HORIZON} cycles?" (FD001, GroupKFold CV)',
    fontsize=11, y=1.02
)
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_classification_prc.png',
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved fig_classification_prc.png')

In [ ]:
print('Classification report (OOF, threshold=0.5):')
print(classification_report(y, oof_pred,
                             target_names=['No failure', f'Fail ≤ {FAIL_HORIZON}']))

---
## 4. Threshold sweep — precision vs recall trade-off

In [ ]:
thresh_range = np.linspace(0.05, 0.95, 50)
pr_data = []
for t in thresh_range:
    preds = (oof_prob >= t).astype(int)
    if preds.sum() == 0:
        continue
    from sklearn.metrics import precision_score, recall_score
    pr_data.append({
        'threshold': t,
        'precision': precision_score(y, preds, zero_division=0),
        'recall':    recall_score(y, preds, zero_division=0),
        'f1':        f1_score(y, preds, zero_division=0),
    })

thresh_df = pd.DataFrame(pr_data)
best_f1_idx = thresh_df['f1'].idxmax()
best_t = thresh_df.loc[best_f1_idx, 'threshold']

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresh_df['threshold'], thresh_df['precision'], label='Precision', color='steelblue', lw=2)
ax.plot(thresh_df['threshold'], thresh_df['recall'],    label='Recall',    color='tomato',    lw=2)
ax.plot(thresh_df['threshold'], thresh_df['f1'],        label='F1',        color='green',     lw=2, ls='--')
ax.axvline(best_t, color='black', ls=':', lw=1.5, label=f'Best F1 threshold={best_t:.2f}')
ax.set_xlabel('Decision threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs classification threshold')
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / 'fig_classification_threshold.png',
            dpi=150, bbox_inches='tight')
plt.show()

print(f'Best F1 threshold: {best_t:.2f}')
print(f'At best threshold — Precision: {thresh_df.loc[best_f1_idx, "precision"]:.3f},',
      f'Recall: {thresh_df.loc[best_f1_idx, "recall"]:.3f},',
      f'F1: {thresh_df.loc[best_f1_idx, "f1"]:.3f}')

---
## 5. Log to MLflow and save final model

In [ ]:
mlflow.set_tracking_uri('file://' + str(Path('../mlruns').resolve()))
mlflow.set_experiment('predictive-maintenance')

with mlflow.start_run(run_name='lgbm_classification_fail30'):
    mlflow.log_param('fail_horizon', FAIL_HORIZON)
    mlflow.log_metric('cv_auc_roc_mean', float(np.mean(fold_auc_roc)))
    mlflow.log_metric('cv_auc_pr_mean',  float(np.mean(fold_auc_pr)))
    mlflow.log_metric('cv_f1_mean',      float(np.mean(fold_f1)))
    mlflow.log_metric('oof_auc_pr',      float(aucpr))

# Train final model on all data
final_clf = LGBMClassifier(
    n_estimators=500, learning_rate=0.05, num_leaves=63,
    class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1,
)
final_clf.fit(X, y)
save_model(final_clf, 'lgbm_classifier_fail30', MODELS_DIR)
print('Final classifier saved.')

---
## 6. Classification vs regression — when to use which?

### When classification is preferable
- The operational decision is binary: "trigger maintenance now or not."
- You have a fixed maintenance horizon (e.g. the next scheduled shift).
- Calibrated probabilities are more useful than a point RUL estimate
  (e.g. "80% probability of failure within 30 cycles").
- The RUL distribution is multimodal (different failure modes produce
  different degradation rates — a single regressor struggles).

### What classification misses that regression captures
- **How urgent is it?** A classifier says "fail soon"; a regressor says
  "fail in 5 cycles" vs "fail in 28 cycles" — very different decisions.
- **Cost optimisation:** the cost-threshold analysis (Notebook 03) requires
  a continuous RUL estimate to sweep thresholds and find the optimal T.
- **Trending:** monitoring RUL over time lets maintenance engineers track
  degradation rate and detect accelerating wear before the alert fires.

### Recommended approach
Use regression as the primary model (it subsumes classification: you can
always threshold the predicted RUL). Add a calibrated classifier on top
for operators who prefer probability outputs.